In [1]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
df = pd.read_csv("dirty_cafe_sales.csv")

print("Number of rows and columns:", df.shape)
df.head(10)

Saving dirty_cafe_sales.csv to dirty_cafe_sales (5).csv
Number of rows and columns: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [2]:
df_original = df.copy()

In [3]:
df.info()
df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


,0
Transaction ID,object
Item,object
Quantity,object
Price Per Unit,object
Total Spent,object
Payment Method,object
Location,object
Transaction Date,object


In [4]:
for col in df.select_dtypes(include='object').columns:
    print(f"\n--- Column: {col} ---")
    print(df[col].unique())


--- Column: Transaction ID ---
['TXN_1961373' 'TXN_4977031' 'TXN_4271903' ... 'TXN_5255387' 'TXN_7695629'
 'TXN_6170729']

--- Column: Item ---
['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']

--- Column: Quantity ---
['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]

--- Column: Price Per Unit ---
['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']

--- Column: Total Spent ---
['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']

--- Column: Payment Method ---
['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]

--- Column: Location ---
['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']

--- Column: Transaction Date ---
['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11'
 '2023-03-31' '2023-10-06' '2023-10-28' '2023-07-28' '2023-12-31'
 '2023-11-07' 'ERROR' '2023-05-03' '2023-06-01' '2023-03-21' '2023-11-15'
 '2023-06-10' '2023-02-24' '202

In [5]:
# List of all possible missing/corrupted value representations found in Step 4
missing_placeholders = ['UNKNOWN', 'ERROR', 'Unknown', 'Error', 'N/A', 'NA',
                         'None', 'none', '-', '', ' ', 'null', 'NULL']

df.replace(missing_placeholders, np.nan, inplace=True)

In [6]:
df.isnull().sum()

,0
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


In [7]:
missing_percent = (df.isnull().sum() / len(df)) * 100
print(missing_percent.sort_values(ascending=False))

Location            39.61
Payment Method      31.78
Item                 9.69
Price Per Unit       5.33
Total Spent          5.02
Quantity             4.79
Transaction Date     4.60
Transaction ID       0.00
dtype: float64


In [8]:
print("Number of duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

Number of duplicate rows: 0


In [12]:
text_columns = ['Item', 'Payment Method', 'Location']

for col in text_columns:
    df[col] = df[col].str.strip()      # remove leading/trailing spaces
    df[col] = df[col].str.title()      # standardize capitalization, e.g., "cash" -> "Cash"

In [13]:
for col in text_columns:
    print(f"\n{col}: {df[col].unique()}")


Item: ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' nan 'Sandwich' 'Juice' 'Tea']

Payment Method: ['Credit Card' 'Cash' nan 'Digital Wallet']

Location: ['Takeaway' 'In-Store' nan]


In [14]:
# Date
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# Numeric
numeric_columns = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [15]:
df.dtypes

,0
Transaction ID,object
Item,object
Quantity,float64
Price Per Unit,float64
Total Spent,float64
Payment Method,object
Location,object
Transaction Date,datetime64[ns]


In [18]:
df['Calculated Total'] = df['Quantity'] * df['Price Per Unit']
mismatch = df[df['Calculated Total'] != df['Total Spent']]

print("Number of rows with inconsistent calculations:", len(mismatch))
mismatch.head()

Number of rows with inconsistent calculations: 58


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Calculated Total
65,TXN_4987129,Sandwich,3.0,NaN,NaN,NaN,In-Store,2023-10-20,NaN
236,TXN_8562645,Salad,NaN,5.0,NaN,NaN,In-Store,2023-05-18,NaN
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15,NaN
629,TXN_9289174,Cake,NaN,NaN,12.0,Digital Wallet,In-Store,2023-12-30,NaN
641,TXN_2962976,Juice,NaN,3.0,NaN,NaN,NaN,2023-03-17,NaN


In [21]:
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11


In [20]:
df = df.drop(columns=['Calculated Total'])  # remove helper column after validation is complete

In [22]:
df = df.dropna(subset=['Quantity', 'Price Per Unit', 'Total Spent'])

In [23]:
df.groupby('Item')['Price Per Unit'].unique()

,Price Per Unit
Item,
Cake,[3.0]
Coffee,[2.0]
Cookie,[1.0]
Juice,[3.0]
Salad,[5.0]
Sandwich,[4.0]
Smoothie,[4.0]
Tea,[1.5]


In [27]:
price_to_item_map = {1.0: 'Cookie', 2.0: 'Coffee', 5.0: 'Salad', 1.5: 'Tea'}
df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(price_to_item_map))

df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,NaN,2023-03-31
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,NaN,2023-10-28
9,TXN_2064365,Sandwich,5.0,4.0,20.0,NaN,In-Store,2023-12-31
10,TXN_2548360,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-11-07
11,TXN_3051279,Sandwich,2.0,4.0,8.0,Credit Card,Takeaway,NaT


In [28]:
df = df.dropna(subset=['Item'])

In [29]:
df['Payment Method'] = df['Payment Method'].fillna(df['Payment Method'].mode()[0])
df['Location'] = df['Location'].fillna(df['Location'].mode()[0])

In [30]:
print("Final shape:", df.shape)
print("\nRemaining missing values:")
print(df.isnull().sum())

print("\nFinal data types:")
print(df.dtypes)

df.head(10)

Final shape: (9468, 8)

Remaining missing values:
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    433
dtype: int64

Final data types:
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,Takeaway,2023-03-31
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,Takeaway,2023-10-28
9,TXN_2064365,Sandwich,5.0,4.0,20.0,Digital Wallet,In-Store,2023-12-31
10,TXN_2548360,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-11-07
11,TXN_3051279,Sandwich,2.0,4.0,8.0,Credit Card,Takeaway,NaT


In [31]:
df = df.dropna(subset=['Transaction Date'])

In [32]:
for col in df.select_dtypes(include='object').columns:
    print(f"\n--- Column: {col} ---")
    print(df[col].unique())


--- Column: Transaction ID ---
['TXN_1961373' 'TXN_4977031' 'TXN_4271903' ... 'TXN_5255387' 'TXN_7695629'
 'TXN_6170729']

--- Column: Item ---
['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'Sandwich' 'Tea' 'Juice']

--- Column: Payment Method ---
['Credit Card' 'Cash' 'Digital Wallet']

--- Column: Location ---
['Takeaway' 'In-Store']


In [34]:
print("=== CLEANING SUMMARY ===")
print(f"Initial number of rows: {len(df_original)}")
print(f"Final number of rows: {len(df)}")
print(f"Rows removed: {len(df_original) - len(df)}")
print(f"Percentage of data remaining: {len(df)/len(df_original)*100:.2f}%")

=== CLEANING SUMMARY ===
Initial number of rows: 10000
Final number of rows: 9035
Rows removed: 965
Percentage of data remaining: 90.35%


In [35]:
df.to_csv("cafe_sales_cleaned_final.csv", index=False)

from google.colab import files
files.download("cafe_sales_cleaned_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>